In [7]:
import os
import numpy as np
import pandas as pd
from google.colab import drive
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

#drive.mount('/content/drive')

NORMAL_PATH = "final_normal.csv"
SHOP_PATH   = "final_shop.csv"

normal = pd.read_csv(NORMAL_PATH)
shop   = pd.read_csv(SHOP_PATH)

exclude_cols = ["video", "frame", "person"]
needed_cols = [c for c in normal.columns if c not in exclude_cols]
print(needed_cols)

def make_seq_df(df, label):
    df = df.sort_values(["video", "person", "frame"])
    rows = []
    for (video, person), g in df.groupby(["video", "person"]):
        seq = g[needed_cols].astype("float32").values.tolist()
        rows.append(((video, person), seq, label))
    return pd.DataFrame(rows, columns=["video_person_id", "pose_seq", "label"])

normal_df = make_seq_df(normal, "normal")
shop_df   = make_seq_df(shop, "shoplifting")
combined_df = pd.concat([normal_df, shop_df], ignore_index=True)

y = (combined_df["label"] == "shoplifting").astype("int32").values

X = pad_sequences(combined_df["pose_seq"], padding="post", dtype="float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_features = X.shape[-1]

model = models.Sequential([
    layers.Masking(mask_value=0.0, input_shape=(None, num_features)),
    layers.GRU(128),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

model.fit(X_train, y_train, validation_split=0.2, epochs=20, batch_size=32)

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype("int32")

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["normal", "shoplifting"]))


['x_0', 'y_0', 'x_1', 'y_1', 'x_2', 'y_2', 'x_3', 'y_3', 'x_4', 'y_4', 'x_5', 'y_5', 'x_6', 'y_6', 'x_7', 'y_7', 'x_8', 'y_8', 'x_9', 'y_9', 'x_10', 'y_10', 'x_11', 'y_11', 'x_12', 'y_12', 'x_13', 'y_13', 'x_14', 'y_14', 'x_15', 'y_15', 'x_16', 'y_16']
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.5507 - auc: 0.4629 - loss: 0.7807 - val_accuracy: 0.5581 - val_auc: 0.4464 - val_loss: 0.6947
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 188ms/step - accuracy: 0.6122 - auc: 0.6673 - loss: 0.6670 - val_accuracy: 0.4651 - val_auc: 0.4369 - val_loss: 0.7304
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 185ms/step - accuracy: 0.6396 - auc: 0.6961 - loss: 0.6374 - val_accuracy: 0.3953 - val_auc: 0.4583 - val_loss: 0.8100
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 277ms/step - accuracy: 0.6144 - auc: 0.6609 - loss: 0.6576 - val_accuracy: 0.4419 - val_auc: 0.4655 - val_loss: 0.7349
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 331ms/step - accuracy: 0.6748 - auc: 0.7339 - loss: 0.6196 - val_accuracy: 0.4419 - val_auc: 0.4143 - val_loss: 0.7482
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 184ms/step - accuracy: 0.6726 - auc: 0.7111 - loss: 0.6245 - val_accuracy: 0.4419 - val_auc: 0.4357 - val_loss: 0.7287
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 181ms/step - accuracy: 0

In [8]:
print("X_train shape:", X_train.shape)

# Analyze person counts per video in the shop dataset
shop_person_counts = shop.groupby("video")["person"].nunique()

print("\nPerson counts per video in shop dataset (showing only those with > 1 person):")
multi_person_videos = shop_person_counts[shop_person_counts > 1]
if not multi_person_videos.empty:
    print(multi_person_videos)
else:
    print("No videos with > 1 person found in the shop dataset.")

print("\nSummary statistics of person counts per video:")
print(shop_person_counts.describe())

X_train shape: (213, 160, 34)

Person counts per video in shop dataset (showing only those with > 1 person):
video
Shoplifting (11).mp4    2
Shoplifting (15).mp4    3
Shoplifting (18).mp4    3
Shoplifting (19).mp4    3
Shoplifting (21).mp4    3
Shoplifting (23).mp4    3
Shoplifting (24).mp4    3
Shoplifting (25).mp4    3
Shoplifting (26).mp4    3
Shoplifting (27).mp4    2
Shoplifting (29).mp4    2
Shoplifting (3).mp4     3
Shoplifting (30).mp4    2
Shoplifting (31).mp4    2
Shoplifting (38).mp4    2
Shoplifting (40).mp4    2
Shoplifting (44).mp4    3
Shoplifting (45).mp4    3
Shoplifting (46).mp4    3
Shoplifting (47).mp4    3
Shoplifting (5).mp4     2
Shoplifting (51).mp4    3
Shoplifting (52).mp4    2
Shoplifting (6).mp4     4
Shoplifting (60).mp4    2
Shoplifting (7).mp4     4
Shoplifting (72).mp4    2
Shoplifting (73).mp4    3
Shoplifting (74).mp4    2
Shoplifting (76).mp4    3
Shoplifting (77).mp4    2
Shoplifting (78).mp4    2
Shoplifting (8).mp4     2
Shoplifting (80).mp4    2
S

## Hyperparameter Tuning

### Subtask:
Perform a grid search over GRU units, dropout rates, and learning rates to optimize model performance.


In [9]:
def build_model(units, dropout_rate, learning_rate):
    model = models.Sequential([
        layers.Masking(mask_value=0.0, input_shape=(None, num_features)),
        layers.GRU(units, dropout=dropout_rate),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

units_list = [64, 128]
dropout_list = [0.2, 0.5]
lr_list = [0.001, 0.0001]

best_acc = 0.0
best_params = {}

print("Starting Grid Search...")
for units in units_list:
    for dropout in dropout_list:
        for lr in lr_list:
            print(f"Testing: Units={units}, Dropout={dropout}, LR={lr}")
            model = build_model(units, dropout, lr)

            # Train the model
            model.fit(
                X_train, y_train,
                validation_split=0.2,
                epochs=15,
                batch_size=32,
                verbose=0
            )

            # Evaluate on test set
            loss, acc = model.evaluate(X_test, y_test, verbose=0)
            print(f"Test Accuracy: {acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_params = {"units": units, "dropout": dropout, "learning_rate": lr}

print("\nBest Hyperparameters found:", best_params)
print(f"Best Test Accuracy: {best_acc:.4f}")

Starting Grid Search...
Testing: Units=64, Dropout=0.2, LR=0.001


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Test Accuracy: 0.4444
Testing: Units=64, Dropout=0.2, LR=0.0001
Test Accuracy: 0.4259
Testing: Units=64, Dropout=0.5, LR=0.001
Test Accuracy: 0.4259
Testing: Units=64, Dropout=0.5, LR=0.0001
Test Accuracy: 0.4259
Testing: Units=128, Dropout=0.2, LR=0.001
Test Accuracy: 0.5370
Testing: Units=128, Dropout=0.2, LR=0.0001
Test Accuracy: 0.5185
Testing: Units=128, Dropout=0.5, LR=0.001
Test Accuracy: 0.5000
Testing: Units=128, Dropout=0.5, LR=0.0001
Test Accuracy: 0.5556

Best Hyperparameters found: {'units': 128, 'dropout': 0.5, 'learning_rate': 0.0001}
Best Test Accuracy: 0.5556


In [10]:
def build_model(units, dropout_rate, learning_rate):
    model = models.Sequential([
        layers.Input(shape=(None, num_features)),
        layers.Masking(mask_value=0.0),
        layers.GRU(units, dropout=dropout_rate),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

units_list = [64, 128]
dropout_list = [0.2, 0.5]
lr_list = [0.001, 0.0001]

best_acc = 0.0
best_params = {}

print("Starting Grid Search...")
for units in units_list:
    for dropout in dropout_list:
        for lr in lr_list:
            print(f"Testing: Units={units}, Dropout={dropout}, LR={lr}")
            model = build_model(units, dropout, lr)

            # Train the model
            model.fit(
                X_train, y_train,
                validation_split=0.2,
                epochs=15,
                batch_size=32,
                verbose=0
            )

            # Evaluate on test set
            loss, acc = model.evaluate(X_test, y_test, verbose=0)
            print(f"Test Accuracy: {acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_params = {"units": units, "dropout": dropout, "learning_rate": lr}

print("\nBest Hyperparameters found:", best_params)
print(f"Best Test Accuracy: {best_acc:.4f}")

Starting Grid Search...
Testing: Units=64, Dropout=0.2, LR=0.001
Test Accuracy: 0.4259
Testing: Units=64, Dropout=0.2, LR=0.0001
Test Accuracy: 0.5926
Testing: Units=64, Dropout=0.5, LR=0.001
Test Accuracy: 0.4630
Testing: Units=64, Dropout=0.5, LR=0.0001
Test Accuracy: 0.4444
Testing: Units=128, Dropout=0.2, LR=0.001
Test Accuracy: 0.4815
Testing: Units=128, Dropout=0.2, LR=0.0001
Test Accuracy: 0.5370
Testing: Units=128, Dropout=0.5, LR=0.001
Test Accuracy: 0.4259
Testing: Units=128, Dropout=0.5, LR=0.0001
Test Accuracy: 0.5185

Best Hyperparameters found: {'units': 64, 'dropout': 0.2, 'learning_rate': 0.0001}
Best Test Accuracy: 0.5926
